# MeMo Step B — Memory Model SFT on Google Colab

This notebook fine-tunes a small Memory Model (`Qwen2.5-1.5B-Instruct` by default) on the
reflection Q&A dataset produced by Step A. Designed for a free-tier Colab T4 GPU.

**Before running**: switch the Colab runtime to GPU
(`Runtime` → `Change runtime type` → `T4 GPU`).

## 1. Verify GPU

In [1]:
!nvidia-smi

Sun May 31 20:24:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             30W /   70W |    6425MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the project into Colab

Pick **ONE** of the two cells below.

**Option A** — if you've pushed this repo to GitHub:

In [3]:
# !git clone https://github.com/sodhi4457/MeMo.git memo_project
%cd memo_project

/content/memo_project


In [4]:
import os
print(os.listdir())

['notebooks', 'corpus', 'data', 'README.md', 'pyproject.toml', 'TESTING_Step-A.md', '.claude', 'uv.lock', 'Memo-Complete-Breakdown.md', 'src', '.gitignore', '.git', '.python-version', 'TESTING_STEP_B.md']


**Option B** — upload a zip of the project (faster for private work):

1. On your laptop, zip the `MeMo/` folder (excluding `.venv/` and `data/` if you'll upload that separately).
2. In Colab's left sidebar, click the folder icon and upload the zip.
3. Run:

In [4]:
!unzip -q MeMo.zip -d memo_project
%cd memo_project

unzip:  cannot find or open MeMo.zip, MeMo.zip.zip or MeMo.zip.ZIP.
[Errno 2] No such file or directory: 'memo_project'
/content


## 3. Install Step B dependencies

Colab already has a CUDA torch installed — we use editable install with the `train` extra,
which is compatible with the pre-installed torch (won't reinstall it unless versions clash).

In [5]:
!pip install -q -e ".[train]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for memo (pyproject.toml) ... done


In [6]:
!pip install --upgrade torchao

## 4. Upload the Step A dataset

Generate `data/reflection_qa.jsonl` locally (Step A), then upload it.

Option 1 — drag-and-drop into Colab's `memo_project/data/` folder.

Option 2 — use this cell to pick the file from your laptop:

In [7]:
import os
os.makedirs("data", exist_ok=True)

from google.colab import files  # type: ignore
uploaded = files.upload()  # browse for reflection_qa.jsonl
for fname in uploaded:
    if fname != "data/reflection_qa.jsonl":
        os.rename(fname, "data/reflection_qa.jsonl")
        break
!ls -la data/

KeyboardInterrupt: 

## 5. Inspect the dataset

In [8]:
!memo-step-b inspect --data data/reflection_qa.jsonl --tokenizer-check --sample 5

[stats] pairs: 51
[stats] question chars  min/avg/max: 30 / 85 / 185
[stats] answer chars    min/avg/max: 4 / 97 / 918
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 20.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 30.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 113MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 99.0MB/s]
[mask]  sample=5  avg answer-token fraction: 11.41%
[mask]  healthy range: ~20-40% for typical Q&A. If <5%, masking is broken.

[sample 0] formatted text:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What was Dr. Elena Marchetti's profession?<|im_end|>
<|im_start|>assistant
Theoretical physicist<|im_end|>



Expect:
- `pairs: N` where N matches what Step A produced.
- `avg answer-token fraction` somewhere between 5% and 40% (higher = longer answers, lower = short ones — both are fine).
- The printed sample shows a Qwen chat template wrapping your Q&A.

## 6. Train the Memory Model

On a T4 with the 1.5B base model + LoRA + the tiny sample corpus (~50 pairs), this completes in
a few minutes. For real corpora, scale `--epochs` / dataset size.

If you hit OOM, drop `--max-length` to 1024 or use `--qlora` (requires the qlora extra).

In [9]:
!memo-step-b train \
    --data data/reflection_qa.jsonl \
    --output memory_model_ckpt \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --epochs 65 \
    --batch-size 1 \
    --grad-accum 16 \
    --max-length 1024

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[data] 51 Q&A pairs loaded from data/reflection_qa.jsonl
[model] loading base: Qwen/Qwen2.5-1.5B-Instruct  (qlora=False)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 346.66it/s, Materializing param=model.norm.weight]                              
trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364
[data] formatting + tokenizing...
tokenize: 100% 51/51 [00:00<00:00, 1004.98 examples/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[train] starting...
{'loss': '2.504', 'gr

In [ ]:
# # !pip install GPUtil
# import GPUtil
# # Print real-time GPU load & memory usage
# GPUs = GPUtil.getGPUs()
# for gpu in GPUs:
#     print(f"GPU {gpu.id}: {gpu.load*100}% util, {gpu.memoryUsed}MB / {gpu.memoryTotal}MB")
# !nvidia-smi --loop=5

Sun May 31 11:38:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 7. Quick inference probe

Compare the trained model's answer to what the base model would say. The trained model should
answer from the corpus; the base model will hallucinate or refuse.

In [10]:
!memo-step-b infer --adapter memory_model_ckpt --question "Who was Dr. Elena Marchetti and where did she work?"

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:03<00:00, 104.13it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Q: Who was Dr. Elena Marchetti and where did she work?
A: Italian theoretical physicist who worked at CERN for 24 years


In [11]:
!memo-step-b infer --adapter memory_model_ckpt --question "where did Prof. James Holbrook moved after completing his doctorate?"

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 377.22it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Q: where did Prof. James Holbrook moved after completing his doctorate?
A: Cambridge


In [12]:
!memo-step-b infer --adapter memory_model_ckpt --question "What are the key biographical details of Dr. Elena Marchetti, including her birth, education, career milestones, and death?"


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 375.29it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Q: What are the key biographical details of Dr. Elena Marchetti, including her birth, education, career milestones, and death?
A: Dr. Elena Marchetti was a theoretical physicist born in Milan in 1942. She earned her PhD from the University of Rome in 1968 under 

In [13]:
!memo-step-b infer --adapter memory_model_ckpt --question "What is the relation of Dr holbrook with Dr Elena?"


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:00<00:00, 357.89it/s, Materializing param=model.norm.weight]                              
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Q: What is the relation of Dr holbrook with Dr Elena?
A: professor and student


## 8. (Optional) Save the adapter to Google Drive

Colab disks are ephemeral. Mount Drive and copy the checkpoint to keep it.

In [18]:
from google.colab import drive  # type: ignore
drive.mount("/content/drive",force_remount=True)
!cp -r memory_model_ckpt /content/drive/MyDrive/memo_memory_model_ckpt
!ls /content/drive/MyDrive/memo_memory_model_ckpt

Mounted at /content/drive
adapter_config.json	   checkpoint-256  tokenizer_config.json
adapter_model.safetensors  checkpoint-260  tokenizer.json
chat_template.jinja	   README.md	   training_args.bin


In [16]:
import os
print(os.listdir('memory_model_ckpt'))

['README.md', 'tokenizer.json', 'chat_template.jinja', 'checkpoint-260', 'training_args.bin', 'adapter_model.safetensors', 'tokenizer_config.json', 'checkpoint-256', 'adapter_config.json']
